# 前置宏定义

In [37]:
%debug off
%define THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 0.005 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define SUB_THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 0.035 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define SUB_THREE_CLOCK2(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 1 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define ADD0C(A,B,CB):(Y,CB)
directive parameters [ k = 2;]
|CB+A->{k}CB+A+Y
|CB+B->{k}CB+B+Y
|CB+Y->{k}CB
%end define

%define ADDCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
//C1先执行，C2后执行,先加和再湮灭
directive parameters [ k = 2;]
%invoke ADD0C(A0,B0,C1):(Y0,C1)
%invoke ADD0C(A1,B1,C1):(Y1,C1)
|C2+Y0+Y1->{k}C2
%end define
%define ADDC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
directive parameters [ k = 2;]
%invoke ADD0C(A0,B0,CB):(Y0,CB)
%invoke ADD0C(A1,B1,CB):(Y1,CB)
|CB+Y0+Y1->{k}CB
%end define

%define MUL0C(A,B,CB):(Y,CB)
directive parameters [ k = 2;]
|A+B+CB->{k}A+B+Y+CB
|Y+CB->{k}CB
%end define

%define MULCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
directive parameters [ k = 2;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
//一阶段计算没有遇到次序问题
%invoke MUL0C(A0,B0,C1):(TMP1,C1)
%invoke MUL0C(A1,B1,C1):(TMP2,C1)
%invoke MUL0C(A0,B1,C1):(TMP3,C1)
%invoke MUL0C(A1,B0,C1):(TMP4,C1)
%invoke ADD0C(TMP1,TMP2,C1):(Y0,C1)
%invoke ADD0C(TMP3,TMP4,C1):(Y1,C1)
|Y0+Y1+C2->{k}C2
%end define

%define MULC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
directive parameters [ k = 2;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
%invoke MUL0C(A0,B0,CB):(TMP1,CB)
%invoke MUL0C(A1,B1,CB):(TMP2,CB)
%invoke MUL0C(A0,B1,CB):(TMP3,CB)
%invoke MUL0C(A1,B0,CB):(TMP4,CB)
%invoke ADD0C(TMP1,TMP2,CB):(Y0,CB)
%invoke ADD0C(TMP3,TMP4,CB):(Y1,CB)
%end define


%define MACC(A0,W0,A1,W1,B0,W2,B1,W3,CB):(Y0,Y1,CB)
directive parameters [ k = 2;]

| 0 Y3
| 0 Y4
| 0 Y5
| 0 Y6
%invoke MULC(A0,A1,W0,W1,CB):(Y3,Y4,CB)
%invoke MULC(B0,B1,W2,W3,CB):(Y5,Y6,CB)
%invoke ADDC(Y3,Y4,Y5,Y6,CB):(Y0,Y1,CB)
%end define

%define TANHC(N0,N1,CB):(H0,H1,CB)
directive parameters [ k = 2]
| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1 c6

%invoke SUB_THREE_CLOCK2(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)

| c3+N0+N1+CB->{k}c3+CB
//正分量逻辑 (对应 dH0/dt = k*N0*(1 - H0^2))
|c5+N0+CB->{k}c5+N0+H0+CB
|N0+H0+H0+c5+CB->{k}c5+N0+H0+CB
|N0+c5+CB->{k}c5+CB
//负分量逻辑 (对应 dH1/dt = k*N1*(1 - H1^2))
|N1+c5+CB->{k}N1+H1+c5+CB
|N1+H1+H1+c5+CB->{k}N1+H1+c5+CB
|N1+c5+CB->{k}c5+CB

%end define

%define NEURON(X0P,X0N,W0P,W0N,X1P,X1N,W1P,W1N,WP,WN,CB):(YP,YN,CB)
//需要三时钟
| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1 c6
| 0 YTMP
| 0 Y0
| 0 Y1
| 0 YTMP0
| 0 YTMP1

%invoke SUB_THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
//第一步，MAC计算
%invoke MACC(X0P,W0P,X0N,W0N,X1P,W1P,X1N,W1N,c1):(Y0,Y1,c1)
//第二步 WP偏置计算
%invoke ADD0C(Y0,WP,c3):(YTMP0,c3)
%invoke ADD0C(Y1,WN,c3):(YTMP1,c3)
//第二步tanh计算
%invoke TANHC(YTMP0,YTMP1,c5):(YP,YN,c5)
%end define

%define XOR_NETWORK(X1P,X1N,X2P,X2N,W01P,W01N,W02P,W02N,W00P,W00N,W11P,W11N,W12P,W12N,W10P,W10N,W21P,W21N,W22P,W22N,W20P,W20N,CB):(c0,c1,CB)

// === 时钟 ===
| 1 clock1 | 1e-6 clock2 | 1e-6 clock3 | 1e-6 clock4 | 1e-6 clock5 | 1 clock6

%invoke THREE_CLOCK(clock1,clock2,clock3,clock4,clock5,clock6,CB):(clock1,clock2,clock3,clock4,clock5,clock6,CB)

// === 神经元调用 ===
%invoke NEURON(X1P, X1N, W01P, W01N, X2P, X2N, W02P, W02N, W00P, W00N, clock1):(H0P, H0N, clock1)
%invoke NEURON(X1P, X1N, W11P, W11N, X2P, X2N, W12P, W12N, W10P, W10N, clock1):(H1P, H1N, clock1)
%invoke NEURON(H0P, H0N, W21P, W21N, H1P, H1N, W22P, W22N, W20P, W20N, clock3):(c0, c1, clock3)

%end define

%define SUB0C(AP,AN,BP,BN,CB):(CP,CN,CB)
%invoke ADD0C(AP,BN,CB):(CP,CB)
%invoke ADD0C(AN,BP,CB):(CN,CB)
%end define

%define SQUAREC(AP,AN,CB):(CP,CN,CB)
%invoke MULC(AP,AN,AP,AN,CB):(CP,CN,CB)
%end define
%define COPY(A):(B)
directive parameters [ k = 2;]
|A->{k}A+B
|B->{k}
%end define
%define LOSSC(YP,YN,TP,TN,CB):(LP,LN,CB)
directive parameters [ k = 2;]
| 0 T0
| 0 T1
%invoke SUB0C(YP,YN,TP,TN,CB):(T0,T1,CB)
%invoke SQUAREC(T0,T1,CB):(LP,LN,CB)
%end define
%define NOISEC(YP,YN,CB):(CP,CN,CB)
| 0.1 WP
%invoke ADD0C(WP,YP,CB):(CP,CB)
%invoke COPY(YN):(CN)
%end define

%define GET_STEPC(YP,YN,TP,TN,CB):(WP,WN,CB)
| 0 W0
| 0 W1
| 0.0055 W2
| 0 W3
%invoke SUB0C(YP,YN,TP,TN,CB):(W0,W1,CB)
%invoke MULC(W0,W1,W2,W3,CB):(WP,WN,CB)
%end define


Debug output disabled.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


# 测试函数

目标函数为$$y=3x^2+2x$$

测试函数权重从0，0开始

In [38]:
%define TEST_FUNC(W0,W1,W2,W3,IP,IN,CB):(OP,ON,CB)
| 0 O1P
| 0 O1N
| 0 O2P
| 0 O2N
| 0 O3P
| 0 O3N
%invoke SQUAREC(IP,IN,CB):(O1P,O1N,CB)
%invoke MULC(W0,W1,O1P,O1N,CB):(O3P,O3N,CB)
%invoke MULC(W2,W3,IP,IN,CB):(O2P,O2N,CB)
%invoke ADD0C(O2P,O3P,CB):(OP,CB)
%invoke ADD0C(O2N,O3N,CB):(ON,CB)
%end define

Debug output disabled.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


还需要进行权重更新

$$ W_{i+1} = W_i - \alpha \times (L(W_i+\Delta W_i)-L(W_i))$$


权重更新函数参数1(另一组权重,原权重，输入，正确输出，当前损失函数结果)


对于顺序无关的计算，DNA/CRN计算确实可以同时进行的，这里是否需要证明是一个交换群？


# 权重更新函数

In [39]:
%define WEIGHTC1(WOP,WON,WiP,WiN,XP,XN,TP,TN,LP,LN,CB):(NWP,NWN,CB)
| 0 c0
| 0 c1
| 0 WP
| 0 WN
| 0 RP
| 0 RN
| 0 L1P
| 0 L1N
| 0 DWP
| 0 DWN
%invoke NOISEC(WiP,WiN,CB):(WP,WN,CB)
%invoke TEST_FUNC(WOP,WON,WP,WN,XP,XN,CB):(RP,RN,CB)
%invoke LOSSC(RP,RN,TP,TN,CB):(L1P,L1N,CB)

%invoke GET_STEPC(L1P,L1N,LP,LN,CB):(DWP,DWN,CB)

%invoke SUB0C(WiP,WiN,DWP,DWN,CB):(NWP,NWN,CB)

%end define 

%define WEIGHTC0(WiP,WiN,WOP,WON,XP,XN,TP,TN,LP,LN,CB):(NWP,NWN,CB)
| 0 c0
| 0 c1
| 0 WP
| 0 WN
| 0 RP
| 0 RN
| 0 L1P
| 0 L1N
| 0 DWP
| 0 DWN
%invoke NOISEC(WiP,WiN,CB):(WP,WN,CB)
%invoke TEST_FUNC(WP,WN,WOP,WON,XP,XN,CB):(RP,RN,CB)
%invoke LOSSC(RP,RN,TP,TN,CB):(L1P,L1N,CB)

%invoke GET_STEPC(L1P,L1N,LP,LN,CB):(DWP,DWN,CB)

%invoke SUB0C(WiP,WiN,DWP,DWN,CB):(NWP,NWN,CB)

%end define 

Debug output disabled.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


# 还需要可复用性

目前大概的思路就是用等量的输入输出物质来进行湮灭，要考虑是否有通用性、互补操作，或者是通过反应物物的互补产生产物的互补？

In [40]:
%define CLEAR(SUBSTANCES, CB):(CB)
directive parameters [ k_clear = 3.0 ]
| SUBSTANCES + CB ->{k_clear} CB
%end define


Debug output disabled.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


### DEEPSEEK生成的数据点（共11个）
| x   | y (真实值) | y (含噪声) |
|-----|-----------|-----------|
| -5  | 65        | 66.5      |
| -4  | 40        | 39.2      |
| -3  | 21        | 20.8      |
| -2  | 8         | 7.5       |
| -1  | 1         | 1.2       |
| 0   | 0         | -0.3      |
| 1   | 5         | 5.1       |
| 2   | 16        | 15.8      |
| 3   | 33        | 33.4      |
| 4   | 56        | 56.7      |
| 5   | 85        | 84.6      |

In [41]:
%reset
%export expanded
%debug on
%title "TEST TRAIN"
directive simulation { 
  final=15; 
  plots=[res1;];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [c0]-[c1];
]	

| 0 W0P
| 0 W0N
| 0 W1P
| 0 W1N

// -5 66.5

| 0 XP
| 5 XN
| 66.5 YP
| 0 YN
| 0 c0
| 0 c1
| 1 CB
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,CB):(c0,c1,CB)


Debug output disabled.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 331
[DEBUG processCode] codeWithoutDefs.Length = 331
[DEBUG processCode] fullExpandedCode.Length = 6676
[DEBUG processC

### Simulation Results

**Simulation Time:** 0.00 to 15.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| TEST_FUNC_0_local_O1P | 25 |
| TEST_FUNC_0_local_SQUAREC_0_local_MULC_0_local_TMP2 | 25 |
| XN | 5 |
| YP | 66.5 |



Simulation completed successfully.

Time range: 0.00 to 15.00
Number of time points: 1001 (requested: 1000)

Number of species: 1

Final concentrations:
  [res1]: 0


# 四时钟

1.前馈计算

2.计算损失函数

3.更新权重

4.清除物质

In [42]:
%define FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
directive parameters [ k_clock = 1 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c7 + CB->{k_clock} c7 + c7+ CB
| c7 + c8 + CB->{k_clock} c8 + c8 +CB
| c8 + c1 + CB->{k_clock} c1 + c1 + CB
%end define

[DEBUG processCode] code.Length = 433
[DEBUG processCode] codeWithoutDefs.Length = 1
[DEBUG processCode] fullExpandedCode.Length = 1
[DEBUG processCode] registry.Instances.Count = 25
[DEBUG processCode] registry.Definitions.Count = 24
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


# 开始测试

In [43]:
%reset
%debug on
%title "TEST TRAIN"
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8

| 10 W0P
| 0 W0N
| 0 W1P
| 0 W1N

//-5 66.5
| 0 XP
| 5 XN
| 66.5 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



[DEBUG processCode] code.Length = 433
[DEBUG processCode] codeWithoutDefs.Length = 1
[DEBUG processCode] fullExpandedCode.Length = 1
[DEBUG processCode] registry.Instances.Count = 25
[DEBUG processCode] registry.Definitions.Count = 24
Macro definitions registered: 24
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 840
[DEBUG processCode] codeWithoutDefs.Length = 840
[DEBUG processCode] fullExpandedCode.Length = 44206
[DEBUG processCode] registry.Instances.Count = 173
[DEBUG processCode] registry.Definitions.Count = 24
[DEBUG] === Simulation Parameters ===
[DEBUG] Simulator: Oslo
[DEBUG] Preserve Mode: OFF (default accumulation)
[DEBUG] Time: initial=0.000000, final=100.000000
[DEBUG] Points: 1000
[DEBUG] Stochastic Scale: 1.000000
[DEBUG] Plots: Key (Rate "res1"), Key (Rate

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 33250 |
| LOSS1P | 66922.2 |
| LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP1 | 62500 |
| LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP2 | 4422.25 |
| LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP3 | 16625 |
| LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP4 | 16625 |
| LOSSC_0_local_T0 | 250 |
| LOSSC_0_local_T1 | 66.5 |
| NW0N | 557.857 |
| NW0P | 562.776 |
| NW1N | 551.314 |
| NW1P | 552.322 |
| R0 | 250 |
| TEST_FUNC_1_local_MULC_4_local_TMP1 | 250 |
| TEST_FUNC_1_local_O1P | 25 |
| TEST_FUNC_1_local_O3P | 250 |
| TEST_FUNC_1_local_SQUAREC_1_local_MULC_3_local_TMP2 | 25 |
| W0P | 10 |
| WEIGHTC0_0_local_DWN | 552.776 |
| WEIGHTC0_0_local_DWP | 557.857 |
| WEIGHTC0_0_local_GET_STEPC_0_local_MULC_11_local_TMP1 | 557.857 |
| WEIGHTC0_0_local_GET_STEPC_0_local_MULC_11_local_TMP4 | 552.776 |
| WEIGHTC0_0_local_GET_STEPC_0_local_W0 | 101428 |
| WEIGHTC0_0_local_GET_STEPC_0_local_W1 | 100505 |
| WEIGHTC0_0_local_GET_STEPC_0_local_W2 | 0.0055 |
| WEIGHTC0_0_local_L1N | 33582.5 |
| WEIGHTC0_0_local_L1P | 68178.5 |
| WEIGHTC0_0_local_LOSSC_1_local_SQUAREC_4_local_MULC_10_local_TMP1 | 63756.3 |
| WEIGHTC0_0_local_LOSSC_1_local_SQUAREC_4_local_MULC_10_local_TMP2 | 4422.25 |
| WEIGHTC0_0_local_LOSSC_1_local_SQUAREC_4_local_MULC_10_local_TMP3 | 16791.2 |
| WEIGHTC0_0_local_LOSSC_1_local_SQUAREC_4_local_MULC_10_local_TMP4 | 16791.2 |
| WEIGHTC0_0_local_LOSSC_1_local_T0 | 252.5 |
| WEIGHTC0_0_local_LOSSC_1_local_T1 | 66.5 |
| WEIGHTC0_0_local_NOISEC_0_local_WP | 0.1 |
| WEIGHTC0_0_local_RP | 252.5 |
| WEIGHTC0_0_local_TEST_FUNC_2_local_MULC_8_local_TMP1 | 252.5 |
| WEIGHTC0_0_local_TEST_FUNC_2_local_O1P | 25 |
| WEIGHTC0_0_local_TEST_FUNC_2_local_O3P | 252.5 |
| WEIGHTC0_0_local_TEST_FUNC_2_local_SQUAREC_3_local_MULC_7_local_TMP2 | 25 |
| WEIGHTC0_0_local_WP | 10.1 |
| WEIGHTC1_0_local_DWN | 552.322 |
| WEIGHTC1_0_local_DWP | 551.314 |
| WEIGHTC1_0_local_GET_STEPC_1_local_MULC_16_local_TMP1 | 551.314 |
| WEIGHTC1_0_local_GET_STEPC_1_local_MULC_16_local_TMP4 | 552.322 |
| WEIGHTC1_0_local_GET_STEPC_1_local_W0 | 100239 |
| WEIGHTC1_0_local_GET_STEPC_1_local_W1 | 100422 |
| WEIGHTC1_0_local_GET_STEPC_1_local_W2 | 0.0055 |
| WEIGHTC1_0_local_L1N | 33500 |
| WEIGHTC1_0_local_L1P | 66989 |
| WEIGHTC1_0_local_LOSSC_2_local_SQUAREC_6_local_MULC_15_local_TMP1 | 62500 |
| WEIGHTC1_0_local_LOSSC_2_local_SQUAREC_6_local_MULC_15_local_TMP2 | 4489 |
| WEIGHTC1_0_local_LOSSC_2_local_SQUAREC_6_local_MULC_15_local_TMP3 | 16750 |
| WEIGHTC1_0_local_LOSSC_2_local_SQUAREC_6_local_MULC_15_local_TMP4 | 16750 |
| WEIGHTC1_0_local_LOSSC_2_local_T0 | 250 |
| WEIGHTC1_0_local_LOSSC_2_local_T1 | 67 |
| WEIGHTC1_0_local_NOISEC_1_local_WP | 0.1 |
| WEIGHTC1_0_local_RN | 0.5 |
| WEIGHTC1_0_local_RP | 250 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_MULC_13_local_TMP1 | 250 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_MULC_14_local_TMP3 | 0.5 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_O1P | 25 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_O2N | 0.5 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_O3P | 250 |
| WEIGHTC1_0_local_TEST_FUNC_3_local_SQUAREC_5_local_MULC_12_local_TMP2 | 25 |
| WEIGHTC1_0_local_WP | 0.1 |
| XN | 5 |
| YP | 66.5 |
| [res1] | 250 |
| [res2] | 33672.3 |
| [res3] | 4.91938 |
| [res4] | 1.00788 |
| c1 | 9.99555e-07 |
| c2 | 9.99415e-07 |
| c3 | 4.99457e-07 |
| c4 | 4.99304e-07 |
| c5 | 0.000726849 |
| c6 | 1.99652 |
| c7 | 0.00275453 |
| c8 | 1.00056e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 250
  [res2]: 33672.3
  [res3]: 4.91938
  [res4]: 1.00788


In [44]:
%preserve NW0P NW0N NW1P NW1N
%debug on
%title "TEST TRAIN"
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

Preserved species set: NW0P, NW0N, NW1P, NW1N
Debug output enabled.
[DEBUG processCode] code.Length = 268
[DEBUG processCode] codeWithoutDefs.Length = 268
[DEBUG processCode] fullExpandedCode.Length = 268
[DEBUG processCode] registry.Instances.Count = 173
[DEBUG processCode] registry.Definitions.Count = 24

=== State from Previous Run (old = last new or 0) ===
  CB: 1.000000
  LOSS1N: 33250.000000
  LOSS1P: 66922.250000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP1: 62500.000000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP2: 4422.250000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP3: 16625.000000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP4: 16625.000000
  LOSSC_0_local_T0: 250.000000
  LOSSC_0_local_T1: 66.500000
  NW0N: 557.856750
  NW0P: 562.776125
  NW1N: 551.314500
  NW1P: 552.322375
  R0: 250.000000
  TEST_FUNC_1_local_MULC_4_local_TMP1: 250.000000
  TEST_FUNC_1_local_O1P: 25.000000
  TEST_FUNC_1_local_O3P: 250.000000
  TEST_FUNC_1_local_SQUAREC_1_local_MULC_3_loc

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| NW0N | 557.857 |
| NW0P | 562.776 |
| NW1N | 551.314 |
| NW1P | 552.322 |
| [res3] | 4.91938 |
| [res4] | 1.00788 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 0
  [res2]: 0
  [res3]: 4.91938
  [res4]: 1.00788


In [45]:
%export expanded
%debug off
%title "TEST TRAIN"
%preserve NW0P NW0N NW1P NW1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [W0P]-[W0N];
  res4 = [W1P]-[W1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8

// -4 39.2
| 0 XP
| 4 XN
| 39.2 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(NW0P,NW0N,NW1P,NW1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 W0P1
| 0 W0N1
| 0 W1P1
| 0 W1N1
%invoke WEIGHTC0(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W0P,W0N,c5)

%invoke WEIGHTC1(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W1P,W1N,c5)



Debug output disabled.
Preserved species set: NW0P, NW0N, NW1P, NW1N
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MULC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MACC(A0, W0, A1, W1, B0, W2, B1, W3, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   TANHC(N0, N1, CB) :(H0, H1, CB) [rate params: ]
//     OK
//   NEURON(X0P, X0N, W0P, W0N, X1P, X

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 1.00032e+09 |
| LOSS1P | 1.00033e+09 |
| LOSSC_3_local_SQUAREC_8_local_MULC_20_local_TMP1 | 5.02627e+08 |
| LOSSC_3_local_SQUAREC_8_local_MULC_20_local_TMP2 | 4.977e+08 |
| LOSSC_3_local_SQUAREC_8_local_MULC_20_local_TMP3 | 5.00158e+08 |
| LOSSC_3_local_SQUAREC_8_local_MULC_20_local_TMP4 | 5.00158e+08 |
| LOSSC_3_local_T0 | 22419.4 |
| LOSSC_3_local_T1 | 22309.2 |
| NW0N | 1115.71 |
| NW0P | 1125.55 |
| NW1N | 1102.63 |
| NW1P | 1104.64 |
| R0 | 22419.4 |
| R1 | 22270 |
| TEST_FUNC_4_local_MULC_18_local_TMP1 | 18008.8 |
| TEST_FUNC_4_local_MULC_18_local_TMP4 | 17851.4 |
| TEST_FUNC_4_local_MULC_19_local_TMP2 | 4410.52 |
| TEST_FUNC_4_local_MULC_19_local_TMP3 | 4418.58 |
| TEST_FUNC_4_local_O1P | 16 |
| TEST_FUNC_4_local_O2N | 4418.58 |
| TEST_FUNC_4_local_O2P | 4410.52 |
| TEST_FUNC_4_local_O3N | 17851.4 |
| TEST_FUNC_4_local_O3P | 18008.8 |
| TEST_FUNC_4_local_SQUAREC_7_local_MULC_17_local_TMP2 | 16 |
| W0N | 1.1005e+07 |
| W0P | 1.10051e+07 |
| W1N | 1.10047e+07 |
| W1P | 1.10047e+07 |
| WEIGHTC0_1_local_DWN | 1.10039e+07 |
| WEIGHTC0_1_local_DWP | 1.10039e+07 |
| WEIGHTC0_1_local_GET_STEPC_2_local_MULC_25_local_TMP1 | 1.10039e+07 |
| WEIGHTC0_1_local_GET_STEPC_2_local_MULC_25_local_TMP4 | 1.10039e+07 |
| WEIGHTC0_1_local_GET_STEPC_2_local_W0 | 2.00071e+09 |
| WEIGHTC0_1_local_GET_STEPC_2_local_W1 | 2.00071e+09 |
| WEIGHTC0_1_local_GET_STEPC_2_local_W2 | 0.0055 |
| WEIGHTC0_1_local_L1N | 1.00039e+09 |
| WEIGHTC0_1_local_L1P | 1.0004e+09 |
| WEIGHTC0_1_local_LOSSC_4_local_SQUAREC_10_local_MULC_24_local_TMP1 | 5.02699e+08 |
| WEIGHTC0_1_local_LOSSC_4_local_SQUAREC_10_local_MULC_24_local_TMP2 | 4.977e+08 |
| WEIGHTC0_1_local_LOSSC_4_local_SQUAREC_10_local_MULC_24_local_TMP3 | 5.00193e+08 |
| WEIGHTC0_1_local_LOSSC_4_local_SQUAREC_10_local_MULC_24_local_TMP4 | 5.00193e+08 |
| WEIGHTC0_1_local_LOSSC_4_local_T0 | 22421 |
| WEIGHTC0_1_local_LOSSC_4_local_T1 | 22309.2 |
| WEIGHTC0_1_local_NOISEC_2_local_WP | 0.1 |
| WEIGHTC0_1_local_RN | 22270 |
| WEIGHTC0_1_local_RP | 22421 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_MULC_22_local_TMP1 | 18010.4 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_MULC_22_local_TMP4 | 17851.4 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_MULC_23_local_TMP2 | 4410.52 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_MULC_23_local_TMP3 | 4418.58 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_O1P | 16 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_O2N | 4418.58 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_O2P | 4410.52 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_O3N | 17851.4 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_O3P | 18010.4 |
| WEIGHTC0_1_local_TEST_FUNC_5_local_SQUAREC_9_local_MULC_21_local_TMP2 | 16 |
| WEIGHTC0_1_local_WN | 1115.71 |
| WEIGHTC0_1_local_WP | 1125.65 |
| WEIGHTC1_1_local_DWN | 1.10036e+07 |
| WEIGHTC1_1_local_DWP | 1.10036e+07 |
| WEIGHTC1_1_local_GET_STEPC_3_local_MULC_30_local_TMP1 | 1.10036e+07 |
| WEIGHTC1_1_local_GET_STEPC_3_local_MULC_30_local_TMP4 | 1.10036e+07 |
| WEIGHTC1_1_local_GET_STEPC_3_local_W0 | 2.00066e+09 |
| WEIGHTC1_1_local_GET_STEPC_3_local_W1 | 2.00066e+09 |
| WEIGHTC1_1_local_GET_STEPC_3_local_W2 | 0.0055 |
| WEIGHTC1_1_local_L1N | 1.00033e+09 |
| WEIGHTC1_1_local_L1P | 1.00035e+09 |
| WEIGHTC1_1_local_LOSSC_5_local_SQUAREC_12_local_MULC_29_local_TMP1 | 5.02627e+08 |
| WEIGHTC1_1_local_LOSSC_5_local_SQUAREC_12_local_MULC_29_local_TMP2 | 4.97718e+08 |
| WEIGHTC1_1_local_LOSSC_5_local_SQUAREC_12_local_MULC_29_local_TMP3 | 5.00167e+08 |
| WEIGHTC1_1_local_LOSSC_5_local_SQUAREC_12_local_MULC_29_local_TMP4 | 5.00167e+08 |
| WEIGHTC1_1_local_LOSSC_5_local_T0 | 22419.4 |
| WEIGHTC1_1_local_LOSSC_5_local_T1 | 22309.6 |
| WEIGHTC1_1_local_NOISEC_3_local_WP | 0.1 |
| WEIGHTC1_1_local_RN | 22270.4 |
| WEIGHTC1_1_local_RP | 22419.4 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_MULC_27_local_TMP1 | 18008.8 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_MULC_27_local_TMP4 | 17851.4 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_MULC_28_local_TMP2 | 4410.52 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_MULC_28_local_TMP3 | 4418.98 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_O1P | 16 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_O2N | 4418.98 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_O2P | 4410.52 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_O3N | 17851.4 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_O3P | 18008.8 |
| WEIGHTC1_1_local_TEST_FUNC_6_local_SQUAREC_11_local_MULC_26_local_TMP2 | 16 |
| WEIGHTC1_1_local_WN | 1102.63 |
| WEIGHTC1_1_local_WP | 1104.74 |
| XN | 4 |
| YP | 39.2 |
| [res1] | 149.357 |
| [res2] | 12134.6 |
| [res3] | 7.88591 |
| [res4] | 2.49956 |
| c1 | 9.99558e-07 |
| c2 | 9.99377e-07 |
| c3 | 4.99448e-07 |
| c4 | 4.99295e-07 |
| c5 | 0.000728296 |
| c6 | 1.99651 |
| c7 | 0.00275979 |
| c8 | 1.00056e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 149.357
  [res2]: 12134.6
  [res3]: 7.88591
  [res4]: 2.49956


In [46]:

%debug off
%title "TEST TRAIN"
%preserve W0P W0N W1P W1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// -3 20.8
| 0 XP
| 3 XN
| 20.8 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



Debug output disabled.
Preserved species set: W0P, W0N, W1P, W1N

=== State from Previous Run (old = last new or 0) ===
  CB: 2.000000
  LOSS1N: 1000348641.083252
  LOSS1P: 1000394447.897901
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP1: 62500.000000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP2: 4422.250000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP3: 16625.000000
  LOSSC_0_local_SQUAREC_2_local_MULC_6_local_TMP4: 16625.000000
  LOSSC_0_local_T0: 250.000000
  LOSSC_0_local_T1: 66.500000
  NW0N: 2231.427000
  NW0P: 2251.104500
  NW1N: 2205.258000
  NW1P: 2209.289500
  R0: 22669.352000
  TEST_FUNC_1_local_MULC_4_local_TMP1: 250.000000
  TEST_FUNC_1_local_O1P: 25.000000
  TEST_FUNC_1_local_O3P: 250.000000
  TEST_FUNC_1_local_SQUAREC_1_local_MULC_3_local_TMP2: 25.000000
  W0P: 11005064.236103
  WEIGHTC0_0_local_DWN: 552.776125
  WEIGHTC0_0_local_DWP: 557.856750
  WEIGHTC0_0_local_GET_STEPC_0_local_MULC_11_local_TMP1: 557.856750
  WEIGHTC0_0_local_GET_STEPC_0_local_MULC_11_lo

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 3.48795e+16 |
| LOSS1P | 3.48795e+16 |
| LOSSC_6_local_SQUAREC_14_local_MULC_34_local_TMP1 | 1.74398e+16 |
| LOSSC_6_local_SQUAREC_14_local_MULC_34_local_TMP2 | 1.74398e+16 |
| LOSSC_6_local_SQUAREC_14_local_MULC_34_local_TMP3 | 1.74398e+16 |
| LOSSC_6_local_SQUAREC_14_local_MULC_34_local_TMP4 | 1.74398e+16 |
| LOSSC_6_local_T0 | 1.3206e+08 |
| LOSSC_6_local_T1 | 1.3206e+08 |
| NW0N | 3.83675e+14 |
| NW0P | 3.83675e+14 |
| NW1N | 3.83675e+14 |
| NW1P | 3.83675e+14 |
| R0 | 1.3206e+08 |
| R1 | 1.3206e+08 |
| TEST_FUNC_7_local_MULC_32_local_TMP1 | 9.90456e+07 |
| TEST_FUNC_7_local_MULC_32_local_TMP4 | 9.90454e+07 |
| TEST_FUNC_7_local_MULC_33_local_TMP2 | 3.30142e+07 |
| TEST_FUNC_7_local_MULC_33_local_TMP3 | 3.30142e+07 |
| TEST_FUNC_7_local_O1P | 9 |
| TEST_FUNC_7_local_O2N | 3.30142e+07 |
| TEST_FUNC_7_local_O2P | 3.30142e+07 |
| TEST_FUNC_7_local_O3N | 9.90454e+07 |
| TEST_FUNC_7_local_O3P | 9.90456e+07 |
| TEST_FUNC_7_local_SQUAREC_13_local_MULC_31_local_TMP2 | 9 |
| W0N | 1.1005e+07 |
| W0P | 1.10051e+07 |
| W1N | 1.10047e+07 |
| W1P | 1.10047e+07 |
| WEIGHTC0_2_local_DWN | 3.83675e+14 |
| WEIGHTC0_2_local_DWP | 3.83675e+14 |
| WEIGHTC0_2_local_GET_STEPC_4_local_MULC_39_local_TMP1 | 3.83675e+14 |
| WEIGHTC0_2_local_GET_STEPC_4_local_MULC_39_local_TMP4 | 3.83675e+14 |
| WEIGHTC0_2_local_GET_STEPC_4_local_W0 | 6.97591e+16 |
| WEIGHTC0_2_local_GET_STEPC_4_local_W1 | 6.97591e+16 |
| WEIGHTC0_2_local_GET_STEPC_4_local_W2 | 0.0055 |
| WEIGHTC0_2_local_L1N | 3.48795e+16 |
| WEIGHTC0_2_local_L1P | 3.48795e+16 |
| WEIGHTC0_2_local_LOSSC_7_local_SQUAREC_16_local_MULC_38_local_TMP1 | 1.74398e+16 |
| WEIGHTC0_2_local_LOSSC_7_local_SQUAREC_16_local_MULC_38_local_TMP2 | 1.74398e+16 |
| WEIGHTC0_2_local_LOSSC_7_local_SQUAREC_16_local_MULC_38_local_TMP3 | 1.74398e+16 |
| WEIGHTC0_2_local_LOSSC_7_local_SQUAREC_16_local_MULC_38_local_TMP4 | 1.74398e+16 |
| WEIGHTC0_2_local_LOSSC_7_local_T0 | 1.3206e+08 |
| WEIGHTC0_2_local_LOSSC_7_local_T1 | 1.3206e+08 |
| WEIGHTC0_2_local_NOISEC_4_local_WP | 0.1 |
| WEIGHTC0_2_local_RN | 1.3206e+08 |
| WEIGHTC0_2_local_RP | 1.3206e+08 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_MULC_36_local_TMP1 | 9.90456e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_MULC_36_local_TMP4 | 9.90454e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_MULC_37_local_TMP2 | 3.30142e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_MULC_37_local_TMP3 | 3.30142e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_O1P | 9 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_O2N | 3.30142e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_O2P | 3.30142e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_O3N | 9.90454e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_O3P | 9.90456e+07 |
| WEIGHTC0_2_local_TEST_FUNC_8_local_SQUAREC_15_local_MULC_35_local_TMP2 | 9 |
| WEIGHTC0_2_local_WN | 1.1005e+07 |
| WEIGHTC0_2_local_WP | 1.10051e+07 |
| WEIGHTC1_2_local_DWN | 3.83675e+14 |
| WEIGHTC1_2_local_DWP | 3.83675e+14 |
| WEIGHTC1_2_local_GET_STEPC_5_local_MULC_44_local_TMP1 | 3.83675e+14 |
| WEIGHTC1_2_local_GET_STEPC_5_local_MULC_44_local_TMP4 | 3.83675e+14 |
| WEIGHTC1_2_local_GET_STEPC_5_local_W0 | 6.97591e+16 |
| WEIGHTC1_2_local_GET_STEPC_5_local_W1 | 6.97591e+16 |
| WEIGHTC1_2_local_GET_STEPC_5_local_W2 | 0.0055 |
| WEIGHTC1_2_local_L1N | 3.48795e+16 |
| WEIGHTC1_2_local_L1P | 3.48795e+16 |
| WEIGHTC1_2_local_LOSSC_8_local_SQUAREC_18_local_MULC_43_local_TMP1 | 1.74398e+16 |
| WEIGHTC1_2_local_LOSSC_8_local_SQUAREC_18_local_MULC_43_local_TMP2 | 1.74398e+16 |
| WEIGHTC1_2_local_LOSSC_8_local_SQUAREC_18_local_MULC_43_local_TMP3 | 1.74398e+16 |
| WEIGHTC1_2_local_LOSSC_8_local_SQUAREC_18_local_MULC_43_local_TMP4 | 1.74398e+16 |
| WEIGHTC1_2_local_LOSSC_8_local_T0 | 1.3206e+08 |
| WEIGHTC1_2_local_LOSSC_8_local_T1 | 1.3206e+08 |
| WEIGHTC1_2_local_NOISEC_5_local_WP | 0.1 |
| WEIGHTC1_2_local_RN | 1.3206e+08 |
| WEIGHTC1_2_local_RP | 1.3206e+08 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_MULC_41_local_TMP1 | 9.90456e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_MULC_41_local_TMP4 | 9.90454e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_MULC_42_local_TMP2 | 3.30142e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_MULC_42_local_TMP3 | 3.30142e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_O1P | 9 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_O2N | 3.30142e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_O2P | 3.30142e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_O3N | 9.90454e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_O3P | 9.90456e+07 |
| WEIGHTC1_2_local_TEST_FUNC_9_local_SQUAREC_17_local_MULC_40_local_TMP2 | 9 |
| WEIGHTC1_2_local_WN | 1.10047e+07 |
| WEIGHTC1_2_local_WP | 1.10047e+07 |
| XN | 3 |
| YP | 20.8 |
| [res1] | 153.474 |
| [res2] | 17600 |
| [res3] | 16.5625 |
| [res4] | 3 |
| c1 | 9.99621e-07 |
| c2 | 9.99377e-07 |
| c3 | 4.9944e-07 |
| c4 | 4.99291e-07 |
| c5 | 0.000727945 |
| c6 | 1.99652 |
| c7 | 0.00275804 |
| c8 | 1.00057e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 153.474
  [res2]: 17600
  [res3]: 16.5625
  [res4]: 3


In [47]:
%export expanded
%debug off
%title "TEST TRAIN"
%preserve NW0P NW0N NW1P NW1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [W0P]-[W0N];
  res4 = [W1P]-[W1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// -2 7.5
| 0 XP
| 2 XN
| 7.5 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(NW0P,NW0N,NW1P,NW1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 W0P
| 0 W0N
| 0 W1P
| 0 W1N
%invoke WEIGHTC0(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W0P,W0N,c5)

%invoke WEIGHTC1(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W1P,W1N,c5)



❌ Simulation: Cannot generate numerical solution


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 Detailed Traceback:

System.ArgumentException: Cannot generate numerical solution
   at CRNKernel.ExecutionEngine.simulate(KernelState kernelState, Crn crn, SimulationConfig config, FSharpList`1 previousState) in d:\Test\crn-2\crn-engine\CRNKernel\ExecutionEngine.fs:line 339


In [48]:

%debug off
%title "TEST TRAIN"
%preserve W0P W0N W1P W1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
//-1  1.2
| 0 XP
| 1 XN
| 1.2 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



Debug output disabled.
Preserved species set: NW0P, NW0N, NW1P, NW1N
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MULC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MACC(A0, W0, A1, W1, B0, W2, B1, W3, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   TANHC(N0, N1, CB) :(H0, H1, CB) [rate params: ]
//     OK
//   NEURON(X0P, X0N, W0P, W0N, X1P, X

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 3.87545e+15 |
| LOSS1P | 3.87545e+15 |
| LOSSC_12_local_SQUAREC_26_local_MULC_62_local_TMP1 | 1.93773e+15 |
| LOSSC_12_local_SQUAREC_26_local_MULC_62_local_TMP2 | 1.93772e+15 |
| LOSSC_12_local_SQUAREC_26_local_MULC_62_local_TMP3 | 1.93772e+15 |
| LOSSC_12_local_SQUAREC_26_local_MULC_62_local_TMP4 | 1.93772e+15 |
| LOSSC_12_local_T0 | 4.40196e+07 |
| LOSSC_12_local_T1 | 4.40196e+07 |
| NW0N | 4.263e+13 |
| NW0P | 4.263e+13 |
| NW1N | 4.263e+13 |
| NW1P | 4.263e+13 |
| R0 | 4.40196e+07 |
| R1 | 4.40196e+07 |
| TEST_FUNC_13_local_MULC_60_local_TMP1 | 2.20101e+07 |
| TEST_FUNC_13_local_MULC_60_local_TMP4 | 2.20101e+07 |
| TEST_FUNC_13_local_MULC_61_local_TMP2 | 2.20095e+07 |
| TEST_FUNC_13_local_MULC_61_local_TMP3 | 2.20095e+07 |
| TEST_FUNC_13_local_O1P | 1 |
| TEST_FUNC_13_local_O2N | 2.20095e+07 |
| TEST_FUNC_13_local_O2P | 2.20095e+07 |
| TEST_FUNC_13_local_O3N | 2.20101e+07 |
| TEST_FUNC_13_local_O3P | 2.20101e+07 |
| TEST_FUNC_13_local_SQUAREC_25_local_MULC_59_local_TMP2 | 1 |
| W0N | 2.20101e+07 |
| W0P | 2.20101e+07 |
| W1N | 2.20095e+07 |
| W1P | 2.20095e+07 |
| WEIGHTC0_4_local_DWN | 4.26299e+13 |
| WEIGHTC0_4_local_DWP | 4.26299e+13 |
| WEIGHTC0_4_local_GET_STEPC_8_local_MULC_67_local_TMP1 | 4.26299e+13 |
| WEIGHTC0_4_local_GET_STEPC_8_local_MULC_67_local_TMP4 | 4.26299e+13 |
| WEIGHTC0_4_local_GET_STEPC_8_local_W0 | 7.7509e+15 |
| WEIGHTC0_4_local_GET_STEPC_8_local_W1 | 7.7509e+15 |
| WEIGHTC0_4_local_GET_STEPC_8_local_W2 | 0.0055 |
| WEIGHTC0_4_local_L1N | 3.87545e+15 |
| WEIGHTC0_4_local_L1P | 3.87545e+15 |
| WEIGHTC0_4_local_LOSSC_13_local_SQUAREC_28_local_MULC_66_local_TMP1 | 1.93773e+15 |
| WEIGHTC0_4_local_LOSSC_13_local_SQUAREC_28_local_MULC_66_local_TMP2 | 1.93772e+15 |
| WEIGHTC0_4_local_LOSSC_13_local_SQUAREC_28_local_MULC_66_local_TMP3 | 1.93772e+15 |
| WEIGHTC0_4_local_LOSSC_13_local_SQUAREC_28_local_MULC_66_local_TMP4 | 1.93772e+15 |
| WEIGHTC0_4_local_LOSSC_13_local_T0 | 4.40196e+07 |
| WEIGHTC0_4_local_LOSSC_13_local_T1 | 4.40196e+07 |
| WEIGHTC0_4_local_NOISEC_8_local_WP | 0.1 |
| WEIGHTC0_4_local_RN | 4.40196e+07 |
| WEIGHTC0_4_local_RP | 4.40196e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_MULC_64_local_TMP1 | 2.20101e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_MULC_64_local_TMP4 | 2.20101e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_MULC_65_local_TMP2 | 2.20095e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_MULC_65_local_TMP3 | 2.20095e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_O1P | 1 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_O2N | 2.20095e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_O2P | 2.20095e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_O3N | 2.20101e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_O3P | 2.20101e+07 |
| WEIGHTC0_4_local_TEST_FUNC_14_local_SQUAREC_27_local_MULC_63_local_TMP2 | 1 |
| WEIGHTC0_4_local_WN | 2.20101e+07 |
| WEIGHTC0_4_local_WP | 2.20101e+07 |
| WEIGHTC1_4_local_DWN | 4.26299e+13 |
| WEIGHTC1_4_local_DWP | 4.26299e+13 |
| WEIGHTC1_4_local_GET_STEPC_9_local_MULC_72_local_TMP1 | 4.26299e+13 |
| WEIGHTC1_4_local_GET_STEPC_9_local_MULC_72_local_TMP4 | 4.26299e+13 |
| WEIGHTC1_4_local_GET_STEPC_9_local_W0 | 7.7509e+15 |
| WEIGHTC1_4_local_GET_STEPC_9_local_W1 | 7.7509e+15 |
| WEIGHTC1_4_local_GET_STEPC_9_local_W2 | 0.0055 |
| WEIGHTC1_4_local_L1N | 3.87545e+15 |
| WEIGHTC1_4_local_L1P | 3.87545e+15 |
| WEIGHTC1_4_local_LOSSC_14_local_SQUAREC_30_local_MULC_71_local_TMP1 | 1.93773e+15 |
| WEIGHTC1_4_local_LOSSC_14_local_SQUAREC_30_local_MULC_71_local_TMP2 | 1.93772e+15 |
| WEIGHTC1_4_local_LOSSC_14_local_SQUAREC_30_local_MULC_71_local_TMP3 | 1.93772e+15 |
| WEIGHTC1_4_local_LOSSC_14_local_SQUAREC_30_local_MULC_71_local_TMP4 | 1.93772e+15 |
| WEIGHTC1_4_local_LOSSC_14_local_T0 | 4.40196e+07 |
| WEIGHTC1_4_local_LOSSC_14_local_T1 | 4.40196e+07 |
| WEIGHTC1_4_local_NOISEC_9_local_WP | 0.1 |
| WEIGHTC1_4_local_RN | 4.40196e+07 |
| WEIGHTC1_4_local_RP | 4.40196e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_MULC_69_local_TMP1 | 2.20101e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_MULC_69_local_TMP4 | 2.20101e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_MULC_70_local_TMP2 | 2.20095e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_MULC_70_local_TMP3 | 2.20095e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_O1P | 1 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_O2N | 2.20095e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_O2P | 2.20095e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_O3N | 2.20101e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_O3P | 2.20101e+07 |
| WEIGHTC1_4_local_TEST_FUNC_15_local_SQUAREC_29_local_MULC_68_local_TMP2 | 1 |
| WEIGHTC1_4_local_WN | 2.20095e+07 |
| WEIGHTC1_4_local_WP | 2.20095e+07 |
| XN | 1 |
| YP | 1.2 |
| [res1] | 30.7727 |
| [res2] | 875 |
| [res3] | 35.7578 |
| [res4] | 5.05469 |
| c1 | 9.99594e-07 |
| c2 | 9.99349e-07 |
| c3 | 4.9947e-07 |
| c4 | 4.99303e-07 |
| c5 | 0.000726516 |
| c6 | 1.99652 |
| c7 | 0.00275327 |
| c8 | 1.00057e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 30.7727
  [res2]: 875
  [res3]: 35.7578
  [res4]: 5.05469


In [49]:
%export expanded
%debug off
%title "TEST TRAIN"
%preserve NW0P NW0N NW1P NW1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [W0P]-[W0N];
  res4 = [W1P]-[W1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// 0 -0.3
| 0 XP
| 0 XN
| 0 YP
| 0.3 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(NW0P,NW0N,NW1P,NW1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 W0P
| 0 W0N
| 0 W1P
| 0 W1N
%invoke WEIGHTC0(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W0P,W0N,c5)

%invoke WEIGHTC1(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W1P,W1N,c5)



In [50]:

%debug off
%title "TEST TRAIN"
%preserve W0P W0N W1P W1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
//1 5.1
| 1 XP
| 0 XN
| 5.1 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



Debug output disabled.
Preserved species set: NW0P, NW0N, NW1P, NW1N
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MULC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MACC(A0, W0, A1, W1, B0, W2, B1, W3, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   TANHC(N0, N1, CB) :(H0, H1, CB) [rate params: ]
//     OK
//   NEURON(X0P, X0N, W0P, W0N, X1P, X

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 1.55018e+16 |
| LOSS1P | 1.55018e+16 |
| LOSSC_18_local_SQUAREC_38_local_MULC_90_local_TMP1 | 7.7509e+15 |
| LOSSC_18_local_SQUAREC_38_local_MULC_90_local_TMP2 | 7.75089e+15 |
| LOSSC_18_local_SQUAREC_38_local_MULC_90_local_TMP3 | 7.7509e+15 |
| LOSSC_18_local_SQUAREC_38_local_MULC_90_local_TMP4 | 7.7509e+15 |
| LOSSC_18_local_T0 | 8.80392e+07 |
| LOSSC_18_local_T1 | 8.80391e+07 |
| NW0N | 1.7052e+14 |
| NW0P | 1.7052e+14 |
| NW1N | 1.7052e+14 |
| NW1P | 1.7052e+14 |
| R0 | 8.80392e+07 |
| R1 | 8.80391e+07 |
| TEST_FUNC_19_local_MULC_88_local_TMP1 | 4.40203e+07 |
| TEST_FUNC_19_local_MULC_88_local_TMP4 | 4.40202e+07 |
| TEST_FUNC_19_local_MULC_89_local_TMP1 | 4.4019e+07 |
| TEST_FUNC_19_local_MULC_89_local_TMP4 | 4.40189e+07 |
| TEST_FUNC_19_local_O1P | 1 |
| TEST_FUNC_19_local_O2N | 4.40189e+07 |
| TEST_FUNC_19_local_O2P | 4.4019e+07 |
| TEST_FUNC_19_local_O3N | 4.40202e+07 |
| TEST_FUNC_19_local_O3P | 4.40203e+07 |
| TEST_FUNC_19_local_SQUAREC_37_local_MULC_87_local_TMP1 | 1 |
| W0N | 4.40202e+07 |
| W0P | 4.40203e+07 |
| W1N | 4.40189e+07 |
| W1P | 4.4019e+07 |
| WEIGHTC0_6_local_DWN | 1.7052e+14 |
| WEIGHTC0_6_local_DWP | 1.7052e+14 |
| WEIGHTC0_6_local_GET_STEPC_12_local_MULC_95_local_TMP1 | 1.7052e+14 |
| WEIGHTC0_6_local_GET_STEPC_12_local_MULC_95_local_TMP4 | 1.7052e+14 |
| WEIGHTC0_6_local_GET_STEPC_12_local_W0 | 3.10036e+16 |
| WEIGHTC0_6_local_GET_STEPC_12_local_W1 | 3.10036e+16 |
| WEIGHTC0_6_local_GET_STEPC_12_local_W2 | 0.0055 |
| WEIGHTC0_6_local_L1N | 1.55018e+16 |
| WEIGHTC0_6_local_L1P | 1.55018e+16 |
| WEIGHTC0_6_local_LOSSC_19_local_SQUAREC_40_local_MULC_94_local_TMP1 | 7.7509e+15 |
| WEIGHTC0_6_local_LOSSC_19_local_SQUAREC_40_local_MULC_94_local_TMP2 | 7.75089e+15 |
| WEIGHTC0_6_local_LOSSC_19_local_SQUAREC_40_local_MULC_94_local_TMP3 | 7.7509e+15 |
| WEIGHTC0_6_local_LOSSC_19_local_SQUAREC_40_local_MULC_94_local_TMP4 | 7.7509e+15 |
| WEIGHTC0_6_local_LOSSC_19_local_T0 | 8.80392e+07 |
| WEIGHTC0_6_local_LOSSC_19_local_T1 | 8.80391e+07 |
| WEIGHTC0_6_local_NOISEC_12_local_WP | 0.1 |
| WEIGHTC0_6_local_RN | 8.80391e+07 |
| WEIGHTC0_6_local_RP | 8.80392e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_MULC_92_local_TMP1 | 4.40203e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_MULC_92_local_TMP4 | 4.40202e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_MULC_93_local_TMP1 | 4.4019e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_MULC_93_local_TMP4 | 4.40189e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_O1P | 1 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_O2N | 4.40189e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_O2P | 4.4019e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_O3N | 4.40202e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_O3P | 4.40203e+07 |
| WEIGHTC0_6_local_TEST_FUNC_20_local_SQUAREC_39_local_MULC_91_local_TMP1 | 1 |
| WEIGHTC0_6_local_WN | 4.40202e+07 |
| WEIGHTC0_6_local_WP | 4.40203e+07 |
| WEIGHTC1_6_local_DWN | 1.7052e+14 |
| WEIGHTC1_6_local_DWP | 1.7052e+14 |
| WEIGHTC1_6_local_GET_STEPC_13_local_MULC_100_local_TMP1 | 1.7052e+14 |
| WEIGHTC1_6_local_GET_STEPC_13_local_MULC_100_local_TMP4 | 1.7052e+14 |
| WEIGHTC1_6_local_GET_STEPC_13_local_W0 | 3.10036e+16 |
| WEIGHTC1_6_local_GET_STEPC_13_local_W1 | 3.10036e+16 |
| WEIGHTC1_6_local_GET_STEPC_13_local_W2 | 0.0055 |
| WEIGHTC1_6_local_L1N | 1.55018e+16 |
| WEIGHTC1_6_local_L1P | 1.55018e+16 |
| WEIGHTC1_6_local_LOSSC_20_local_SQUAREC_42_local_MULC_99_local_TMP1 | 7.7509e+15 |
| WEIGHTC1_6_local_LOSSC_20_local_SQUAREC_42_local_MULC_99_local_TMP2 | 7.75089e+15 |
| WEIGHTC1_6_local_LOSSC_20_local_SQUAREC_42_local_MULC_99_local_TMP3 | 7.7509e+15 |
| WEIGHTC1_6_local_LOSSC_20_local_SQUAREC_42_local_MULC_99_local_TMP4 | 7.7509e+15 |
| WEIGHTC1_6_local_LOSSC_20_local_T0 | 8.80392e+07 |
| WEIGHTC1_6_local_LOSSC_20_local_T1 | 8.80391e+07 |
| WEIGHTC1_6_local_NOISEC_13_local_WP | 0.1 |
| WEIGHTC1_6_local_RN | 8.80391e+07 |
| WEIGHTC1_6_local_RP | 8.80392e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_MULC_97_local_TMP1 | 4.40203e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_MULC_97_local_TMP4 | 4.40202e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_MULC_98_local_TMP1 | 4.4019e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_MULC_98_local_TMP4 | 4.40189e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_O1P | 1 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_O2N | 4.40189e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_O2P | 4.4019e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_O3N | 4.40202e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_O3P | 4.40203e+07 |
| WEIGHTC1_6_local_TEST_FUNC_21_local_SQUAREC_41_local_MULC_96_local_TMP1 | 1 |
| WEIGHTC1_6_local_WN | 4.40189e+07 |
| WEIGHTC1_6_local_WP | 4.4019e+07 |
| XP | 1 |
| YP | 5.1 |
| [res1] | 81.5419 |
| [res2] | 5850 |
| [res3] | 71.5312 |
| [res4] | 9.96875 |
| c1 | 9.9958e-07 |
| c2 | 9.9934e-07 |
| c3 | 4.99456e-07 |
| c4 | 4.99299e-07 |
| c5 | 0.000726665 |
| c6 | 1.99652 |
| c7 | 0.00275342 |
| c8 | 1.00054e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 81.5419
  [res2]: 5850
  [res3]: 71.5312
  [res4]: 9.96875


In [51]:
%export expanded
%debug off
%title "TEST TRAIN"
%preserve NW0P NW0N NW1P NW1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [W0P]-[W0N];
  res4 = [W1P]-[W1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
//2 15.8 
| 2 XP
| 0 XN
| 15.8 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(NW0P,NW0N,NW1P,NW1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 W0P
| 0 W0N
| 0 W1P
| 0 W1N
%invoke WEIGHTC0(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W0P,W0N,c5)

%invoke WEIGHTC1(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W1P,W1N,c5)



In [52]:

%debug off
%title "TEST TRAIN"
%preserve W0P W0N W1P W1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// 3 33.4
| 3 XP
| 0 XN
| 33.4 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



Debug output disabled.
Preserved species set: NW0P, NW0N, NW1P, NW1N
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MULC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MACC(A0, W0, A1, W1, B0, W2, B1, W3, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   TANHC(N0, N1, CB) :(H0, H1, CB) [rate params: ]
//     OK
//   NEURON(X0P, X0N, W0P, W0N, X1P, X

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 2.23229e+18 |
| LOSS1P | 2.23229e+18 |
| LOSSC_24_local_SQUAREC_50_local_MULC_118_local_TMP1 | 1.11615e+18 |
| LOSSC_24_local_SQUAREC_50_local_MULC_118_local_TMP2 | 1.11614e+18 |
| LOSSC_24_local_SQUAREC_50_local_MULC_118_local_TMP3 | 1.11615e+18 |
| LOSSC_24_local_SQUAREC_50_local_MULC_118_local_TMP4 | 1.11615e+18 |
| LOSSC_24_local_T0 | 1.05648e+09 |
| LOSSC_24_local_T1 | 1.05648e+09 |
| NW0N | 2.45552e+16 |
| NW0P | 2.45552e+16 |
| NW1N | 2.45552e+16 |
| NW1P | 2.45552e+16 |
| R0 | 1.05648e+09 |
| R1 | 1.05648e+09 |
| TEST_FUNC_25_local_MULC_116_local_TMP1 | 7.92365e+08 |
| TEST_FUNC_25_local_MULC_116_local_TMP4 | 7.92363e+08 |
| TEST_FUNC_25_local_MULC_117_local_TMP1 | 2.64114e+08 |
| TEST_FUNC_25_local_MULC_117_local_TMP4 | 2.64114e+08 |
| TEST_FUNC_25_local_O1P | 9 |
| TEST_FUNC_25_local_O2N | 2.64114e+08 |
| TEST_FUNC_25_local_O2P | 2.64114e+08 |
| TEST_FUNC_25_local_O3N | 7.92363e+08 |
| TEST_FUNC_25_local_O3P | 7.92365e+08 |
| TEST_FUNC_25_local_SQUAREC_49_local_MULC_115_local_TMP1 | 9 |
| W0N | 8.80404e+07 |
| W0P | 8.80405e+07 |
| W1N | 8.80379e+07 |
| W1P | 8.80379e+07 |
| WEIGHTC0_8_local_DWN | 2.45552e+16 |
| WEIGHTC0_8_local_DWP | 2.45552e+16 |
| WEIGHTC0_8_local_GET_STEPC_16_local_MULC_123_local_TMP1 | 2.45552e+16 |
| WEIGHTC0_8_local_GET_STEPC_16_local_MULC_123_local_TMP4 | 2.45552e+16 |
| WEIGHTC0_8_local_GET_STEPC_16_local_W0 | 4.46458e+18 |
| WEIGHTC0_8_local_GET_STEPC_16_local_W1 | 4.46458e+18 |
| WEIGHTC0_8_local_GET_STEPC_16_local_W2 | 0.0055 |
| WEIGHTC0_8_local_L1N | 2.23229e+18 |
| WEIGHTC0_8_local_L1P | 2.23229e+18 |
| WEIGHTC0_8_local_LOSSC_25_local_SQUAREC_52_local_MULC_122_local_TMP1 | 1.11615e+18 |
| WEIGHTC0_8_local_LOSSC_25_local_SQUAREC_52_local_MULC_122_local_TMP2 | 1.11614e+18 |
| WEIGHTC0_8_local_LOSSC_25_local_SQUAREC_52_local_MULC_122_local_TMP3 | 1.11615e+18 |
| WEIGHTC0_8_local_LOSSC_25_local_SQUAREC_52_local_MULC_122_local_TMP4 | 1.11615e+18 |
| WEIGHTC0_8_local_LOSSC_25_local_T0 | 1.05648e+09 |
| WEIGHTC0_8_local_LOSSC_25_local_T1 | 1.05648e+09 |
| WEIGHTC0_8_local_NOISEC_16_local_WP | 0.1 |
| WEIGHTC0_8_local_RN | 1.05648e+09 |
| WEIGHTC0_8_local_RP | 1.05648e+09 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_MULC_120_local_TMP1 | 7.92365e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_MULC_120_local_TMP4 | 7.92363e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_MULC_121_local_TMP1 | 2.64114e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_MULC_121_local_TMP4 | 2.64114e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_O1P | 9 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_O2N | 2.64114e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_O2P | 2.64114e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_O3N | 7.92363e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_O3P | 7.92365e+08 |
| WEIGHTC0_8_local_TEST_FUNC_26_local_SQUAREC_51_local_MULC_119_local_TMP1 | 9 |
| WEIGHTC0_8_local_WN | 8.80404e+07 |
| WEIGHTC0_8_local_WP | 8.80405e+07 |
| WEIGHTC1_8_local_DWN | 2.45552e+16 |
| WEIGHTC1_8_local_DWP | 2.45552e+16 |
| WEIGHTC1_8_local_GET_STEPC_17_local_MULC_128_local_TMP1 | 2.45552e+16 |
| WEIGHTC1_8_local_GET_STEPC_17_local_MULC_128_local_TMP4 | 2.45552e+16 |
| WEIGHTC1_8_local_GET_STEPC_17_local_W0 | 4.46458e+18 |
| WEIGHTC1_8_local_GET_STEPC_17_local_W1 | 4.46458e+18 |
| WEIGHTC1_8_local_GET_STEPC_17_local_W2 | 0.0055 |
| WEIGHTC1_8_local_L1N | 2.23229e+18 |
| WEIGHTC1_8_local_L1P | 2.23229e+18 |
| WEIGHTC1_8_local_LOSSC_26_local_SQUAREC_54_local_MULC_127_local_TMP1 | 1.11615e+18 |
| WEIGHTC1_8_local_LOSSC_26_local_SQUAREC_54_local_MULC_127_local_TMP2 | 1.11614e+18 |
| WEIGHTC1_8_local_LOSSC_26_local_SQUAREC_54_local_MULC_127_local_TMP3 | 1.11615e+18 |
| WEIGHTC1_8_local_LOSSC_26_local_SQUAREC_54_local_MULC_127_local_TMP4 | 1.11615e+18 |
| WEIGHTC1_8_local_LOSSC_26_local_T0 | 1.05648e+09 |
| WEIGHTC1_8_local_LOSSC_26_local_T1 | 1.05648e+09 |
| WEIGHTC1_8_local_NOISEC_17_local_WP | 0.1 |
| WEIGHTC1_8_local_RN | 1.05648e+09 |
| WEIGHTC1_8_local_RP | 1.05648e+09 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_MULC_125_local_TMP1 | 7.92365e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_MULC_125_local_TMP4 | 7.92363e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_MULC_126_local_TMP1 | 2.64114e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_MULC_126_local_TMP4 | 2.64114e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_O1P | 9 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_O2N | 2.64114e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_O2P | 2.64114e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_O3N | 7.92363e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_O3P | 7.92365e+08 |
| WEIGHTC1_8_local_TEST_FUNC_27_local_SQUAREC_53_local_MULC_124_local_TMP1 | 9 |
| WEIGHTC1_8_local_WN | 8.80379e+07 |
| WEIGHTC1_8_local_WP | 8.80379e+07 |
| XP | 3 |
| YP | 33.4 |
| [res1] | 1347.77 |
| [res2] | 1.72749e+06 |
| [res3] | 124 |
| [res4] | 12 |
| c1 | 9.99582e-07 |
| c2 | 9.99368e-07 |
| c3 | 4.99439e-07 |
| c4 | 4.99282e-07 |
| c5 | 0.000728183 |
| c6 | 1.99652 |
| c7 | 0.00275845 |
| c8 | 1.00054e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 1347.77
  [res2]: 1.72749e+06
  [res3]: 124
  [res4]: 12


In [53]:
%export expanded
%debug off
%title "TEST TRAIN"
%preserve NW0P NW0N NW1P NW1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [W0P]-[W0N];
  res4 = [W1P]-[W1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// 4 56.7
| 4 XP
| 0 XN
| 56.7 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(NW0P,NW0N,NW1P,NW1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 W0P
| 0 W0N
| 0 W1P
| 0 W1N
%invoke WEIGHTC0(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W0P,W0N,c5)

%invoke WEIGHTC1(NW0P,NW0N,NW1P,NW1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(W1P,W1N,c5)



In [54]:

%debug off
%title "TEST TRAIN"
%preserve W0P W0N W1P W1N
directive simulation { 
  final=100; 
  plots=[res1;res2;res3;res4];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [R0]-[R1];
  res2 = [LOSS1P]-[LOSS1N];
  res3 = [NW0P]-[NW0N];
  res4 = [NW1P]-[NW1N];
]	

| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1e-6 c6
| 1e-6 c7
| 1 c8
// 5 84.6
| 5 XP
| 0 XN
| 84.6 YP
| 0 YN
| 0 R0
| 0 R1
| 1 CB
%invoke FOUR_CLOCK(c1,c2,c3,c4,c5,c6,c7,c8,CB):(c1,c2,c3,c4,c5,c6,c7,c8,CB)
%invoke TEST_FUNC(W0P,W0N,W1P,W1N,XP,XN,c1):(R0,R1,CB)

| 0 LOSS1P
| 0 LOSS1N

%invoke LOSSC(R0,R1,YP,YN,c3):(LOSS1P,LOSS1N,c3)

| 0 NW0P
| 0 NW0N
%invoke WEIGHTC0(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW0P,NW0N,c5)

| 0 NW1P
| 0 NW1N
%invoke WEIGHTC1(W0P,W0N,W1P,W1N,XP,XN,YP,YN,LOSS1P,LOSS1N,c5):(NW1P,NW1N,c5)



Debug output disabled.
Preserved species set: NW0P, NW0N, NW1P, NW1N
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MULC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MACC(A0, W0, A1, W1, B0, W2, B1, W3, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   TANHC(N0, N1, CB) :(H0, H1, CB) [rate params: ]
//     OK
//   NEURON(X0P, X0N, W0P, W0N, X1P, X

### Simulation Results

**Simulation Time:** 0.00 to 100.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| CB | 1 |
| LOSS1N | 5.58075e+19 |
| LOSS1P | 5.58075e+19 |
| LOSSC_30_local_SQUAREC_62_local_MULC_146_local_TMP1 | 2.79038e+19 |
| LOSSC_30_local_SQUAREC_62_local_MULC_146_local_TMP2 | 2.79037e+19 |
| LOSSC_30_local_SQUAREC_62_local_MULC_146_local_TMP3 | 2.79038e+19 |
| LOSSC_30_local_SQUAREC_62_local_MULC_146_local_TMP4 | 2.79038e+19 |
| LOSSC_30_local_T0 | 5.2824e+09 |
| LOSSC_30_local_T1 | 5.2824e+09 |
| NW0N | 6.13883e+17 |
| NW0P | 6.13883e+17 |
| NW1N | 6.13883e+17 |
| NW1P | 6.13883e+17 |
| R0 | 5.2824e+09 |
| R1 | 5.2824e+09 |
| TEST_FUNC_31_local_MULC_144_local_TMP1 | 4.40203e+09 |
| TEST_FUNC_31_local_MULC_144_local_TMP4 | 4.40202e+09 |
| TEST_FUNC_31_local_MULC_145_local_TMP1 | 8.80379e+08 |
| TEST_FUNC_31_local_MULC_145_local_TMP4 | 8.80379e+08 |
| TEST_FUNC_31_local_O1P | 25 |
| TEST_FUNC_31_local_O2N | 8.80379e+08 |
| TEST_FUNC_31_local_O2P | 8.80379e+08 |
| TEST_FUNC_31_local_O3N | 4.40202e+09 |
| TEST_FUNC_31_local_O3P | 4.40203e+09 |
| TEST_FUNC_31_local_SQUAREC_61_local_MULC_143_local_TMP1 | 25 |
| W0N | 1.76081e+08 |
| W0P | 1.76081e+08 |
| W1N | 1.76076e+08 |
| W1P | 1.76076e+08 |
| WEIGHTC0_10_local_DWN | 6.13883e+17 |
| WEIGHTC0_10_local_DWP | 6.13883e+17 |
| WEIGHTC0_10_local_GET_STEPC_20_local_MULC_151_local_TMP1 | 6.13883e+17 |
| WEIGHTC0_10_local_GET_STEPC_20_local_MULC_151_local_TMP4 | 6.13883e+17 |
| WEIGHTC0_10_local_GET_STEPC_20_local_W0 | 1.11615e+20 |
| WEIGHTC0_10_local_GET_STEPC_20_local_W1 | 1.11615e+20 |
| WEIGHTC0_10_local_GET_STEPC_20_local_W2 | 0.0055 |
| WEIGHTC0_10_local_L1N | 5.58075e+19 |
| WEIGHTC0_10_local_L1P | 5.58075e+19 |
| WEIGHTC0_10_local_LOSSC_31_local_SQUAREC_64_local_MULC_150_local_TMP1 | 2.79038e+19 |
| WEIGHTC0_10_local_LOSSC_31_local_SQUAREC_64_local_MULC_150_local_TMP2 | 2.79037e+19 |
| WEIGHTC0_10_local_LOSSC_31_local_SQUAREC_64_local_MULC_150_local_TMP3 | 2.79038e+19 |
| WEIGHTC0_10_local_LOSSC_31_local_SQUAREC_64_local_MULC_150_local_TMP4 | 2.79038e+19 |
| WEIGHTC0_10_local_LOSSC_31_local_T0 | 5.2824e+09 |
| WEIGHTC0_10_local_LOSSC_31_local_T1 | 5.2824e+09 |
| WEIGHTC0_10_local_NOISEC_20_local_WP | 0.1 |
| WEIGHTC0_10_local_RN | 5.2824e+09 |
| WEIGHTC0_10_local_RP | 5.2824e+09 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_MULC_148_local_TMP1 | 4.40203e+09 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_MULC_148_local_TMP4 | 4.40202e+09 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_MULC_149_local_TMP1 | 8.80379e+08 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_MULC_149_local_TMP4 | 8.80379e+08 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_O1P | 25 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_O2N | 8.80379e+08 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_O2P | 8.80379e+08 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_O3N | 4.40202e+09 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_O3P | 4.40203e+09 |
| WEIGHTC0_10_local_TEST_FUNC_32_local_SQUAREC_63_local_MULC_147_local_TMP1 | 25 |
| WEIGHTC0_10_local_WN | 1.76081e+08 |
| WEIGHTC0_10_local_WP | 1.76081e+08 |
| WEIGHTC1_10_local_DWN | 6.13883e+17 |
| WEIGHTC1_10_local_DWP | 6.13883e+17 |
| WEIGHTC1_10_local_GET_STEPC_21_local_MULC_156_local_TMP1 | 6.13883e+17 |
| WEIGHTC1_10_local_GET_STEPC_21_local_MULC_156_local_TMP4 | 6.13883e+17 |
| WEIGHTC1_10_local_GET_STEPC_21_local_W0 | 1.11615e+20 |
| WEIGHTC1_10_local_GET_STEPC_21_local_W1 | 1.11615e+20 |
| WEIGHTC1_10_local_GET_STEPC_21_local_W2 | 0.0055 |
| WEIGHTC1_10_local_L1N | 5.58075e+19 |
| WEIGHTC1_10_local_L1P | 5.58075e+19 |
| WEIGHTC1_10_local_LOSSC_32_local_SQUAREC_66_local_MULC_155_local_TMP1 | 2.79038e+19 |
| WEIGHTC1_10_local_LOSSC_32_local_SQUAREC_66_local_MULC_155_local_TMP2 | 2.79037e+19 |
| WEIGHTC1_10_local_LOSSC_32_local_SQUAREC_66_local_MULC_155_local_TMP3 | 2.79038e+19 |
| WEIGHTC1_10_local_LOSSC_32_local_SQUAREC_66_local_MULC_155_local_TMP4 | 2.79038e+19 |
| WEIGHTC1_10_local_LOSSC_32_local_T0 | 5.2824e+09 |
| WEIGHTC1_10_local_LOSSC_32_local_T1 | 5.2824e+09 |
| WEIGHTC1_10_local_NOISEC_21_local_WP | 0.1 |
| WEIGHTC1_10_local_RN | 5.2824e+09 |
| WEIGHTC1_10_local_RP | 5.2824e+09 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_MULC_153_local_TMP1 | 4.40203e+09 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_MULC_153_local_TMP4 | 4.40202e+09 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_MULC_154_local_TMP1 | 8.80379e+08 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_MULC_154_local_TMP4 | 8.80379e+08 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_O1P | 25 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_O2N | 8.80379e+08 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_O2P | 8.80379e+08 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_O3N | 4.40202e+09 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_O3P | 4.40203e+09 |
| WEIGHTC1_10_local_TEST_FUNC_33_local_SQUAREC_65_local_MULC_152_local_TMP1 | 25 |
| WEIGHTC1_10_local_WN | 1.76076e+08 |
| WEIGHTC1_10_local_WP | 1.76076e+08 |
| XP | 5 |
| YP | 84.6 |
| [res1] | 7354.33 |
| [res2] | 5.28548e+07 |
| c1 | 9.99593e-07 |
| c2 | 9.99373e-07 |
| c3 | 4.99449e-07 |
| c4 | 4.99317e-07 |
| c5 | 0.00072739 |
| c6 | 1.99652 |
| c7 | 0.00275573 |
| c8 | 1.00055e-06 |



Simulation completed successfully.

Time range: 0.00 to 100.00
Number of time points: 1001 (requested: 1000)

Number of species: 4

Final concentrations:
  [res1]: 7354.33
  [res2]: 5.28548e+07
  [res3]: 0
  [res4]: 0


# 最后结果